# Análisis Exploratorio — NHANES Cardiometabólico

Exploración del dataset integrado (`data/03_primary/cardiometabolic.parquet`) generado por el pipeline ETL.

**Objetivo:** entender la distribución de factores de riesgo cardiometabólico (obesidad, hipertensión, diabetes) en la cohorte adulta de NHANES 2017-2018.

In [ ]:
import pandas as pd
import plotly.express as px

pd.set_option('display.max_columns', 60)
df = pd.read_parquet('../data/03_primary/cardiometabolic.parquet')
print(f'Filas: {len(df):,}  |  Columnas: {df.shape[1]}')
df.head()

## 1. Calidad de datos: valores faltantes

In [ ]:
nulos = (df.isna().mean() * 100).round(1).sort_values(ascending=False)
nulos[nulos > 0].head(20)

## 2. Cohorte de análisis: adultos 18+

El análisis clínico se define sobre adultos. Filtramos menores y edad desconocida.

In [ ]:
adultos = df[df['age_group'].isin(['18-29', '30-44', '45-64', '65+'])].copy()
print(f'Adultos: {len(adultos):,} de {len(df):,} registros')
adultos[['age', 'bmi', 'bp_systolic_mean', 'glucose_mgdl', 'hba1c_pct']].describe().round(1)

## 3. Distribución del IMC por categoría

In [ ]:
orden = ['Bajo peso', 'Normal', 'Sobrepeso', 'Obesidad']
fig = px.histogram(adultos, x='bmi', nbins=50, title='Distribución del IMC (adultos)')
fig.add_vline(x=25, line_dash='dash'); fig.add_vline(x=30, line_dash='dash')
fig.show()
adultos['bmi_category'].value_counts().reindex(orden)

## 4. Prevalencia de factores de riesgo por grupo etario

In [ ]:
flags = ['obesity_flag', 'hypertension_flag', 'diabetes_flag', 'smoker_flag']
prev = adultos.groupby('age_group')[flags].mean().mul(100).round(1)
prev = prev.reindex(['18-29', '30-44', '45-64', '65+'])
fig = px.bar(prev, barmode='group', title='Prevalencia (%) por grupo etario')
fig.show()
prev

## 5. Correlación entre variables clínicas

In [ ]:
num = ['age', 'bmi', 'waist_cm', 'bp_systolic_mean', 'bp_diastolic_mean',
       'glucose_mgdl', 'hba1c_pct', 'cholesterol_total', 'cholesterol_hdl', 'triglycerides']
num = [c for c in num if c in adultos.columns]
corr = adultos[num].corr().round(2)
px.imshow(corr, text_auto=True, aspect='auto', title='Matriz de correlación').show()

## 6. Score de riesgo cardiometabólico

In [ ]:
px.histogram(adultos, x='cardiometabolic_risk', color='sex',
             title='Distribución del score de riesgo (0-4) por sexo', barmode='group').show()
adultos['cardiometabolic_risk'].value_counts().sort_index()

## Conclusiones

- La prevalencia de **hipertensión** y **diabetes** aumenta marcadamente con la edad.
- La **obesidad** es alta en todos los grupos adultos (~32-42%).
- El **IMC**, la **circunferencia de cintura** y la **glucosa** muestran correlación positiva esperada.
- El **score de riesgo** concentra los valores altos en adultos de 45+ años.

Estos hallazgos alimentan las visualizaciones del dashboard por audiencia.